# Heretic Colab Setup
このノートブックは Heretic を Google Colab 上で動かします。GPU を選択し、必要な依存をインストールしてからモデルを実行します。

In [ ]:
#@title Optional: Mount Google Drive (チェックポイント保存のみ)
# Drive に依存したくない場合は False にしてください（デフォルト: False）。
MOUNT_DRIVE = False  #@param {type:"boolean"}

if MOUNT_DRIVE:
    from google.colab import drive
    import os
    print('Mounting Google Drive...')
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/heretic'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print('Drive mounted at /content/drive. Using', DRIVE_ROOT)
else:
    print('Google Drive mounting skipped. Checkpoints will default to /content/checkpoints unless you change settings below.')


In [1]:
# ランタイム確認（GPU が利用可能か確認し、なければ対処法を表示します）
import shutil, sys, textwrap
if shutil.which('nvidia-smi') is None:
    print(textwrap.dedent('''
nvidia-smi が見つかりません。GPU が有効なランタイムを選択してください。
'''))
else:
    # GPU ドライバ情報を表示
    !nvidia-smi

Sat Apr 18 07:39:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# pip を最新化して依存を入れる（CUDA バージョンに合わせて index-url を変更してください）
!pip install pip -q
!pip install uv -q

!uv self version

uv 0.11.7 (x86_64-unknown-linux-gnu)


In [ ]:
# # 例: cu121 を使用する場合
# !uv pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
# !uv pip install --upgrade bitsandbytes -q
# !uv pip install --upgrade transformers -q
# # !uv pip install git+https://github.com/p-e-w/heretic.git -q
# # もしくはローカル repo をアップロードして uv pip install -e /path/to/repo

# !uv pip list | grep -E "torch|transformers|bitsandbytes"

^C
Using Python 3.12.13 environment at: /usr
sentence-transformers                    5.4.0
torch                                    2.10.0+cpu
torchao                                  0.10.0
torchaudio                               2.10.0+cpu
torchcodec                               0.10.0
torchdata                                0.11.0
torchsummary                             1.5.1
torchtune                                0.6.1
torchvision                              0.25.0+cpu
transformers                             5.5.4


In [ ]:
#@title Clone & install Heretic (フォームで Repo/Branch を指定)
# フォーム: `repo`, `branch`, `dest` を指定してください。実行するとクローンして `pip install -e` を行います。
repo = "Akikukeo1/Live-Vision-Narrator"  #@param {type:"string"}
branch = "main"  #@param {type:"string"}
dest = "/content/Live-Vision-Narrator"  #@param {type:"string"}

import subprocess, importlib.util, gc, os, sys

print(f'Cloning https://github.com/{repo}.git (branch {branch}) into {dest} ...')
try:
    # depth=1 で軽量にクローン。既に存在する場合は fetch/checkout を試みる
    if os.path.exists(dest):
        print('既に存在するため fetch+checkout を試みます')
        subprocess.run(['git','-C',dest,'fetch','--depth','1','origin',branch], check=False)
        subprocess.run(['git','-C',dest,'checkout',branch], check=False)
        subprocess.run(['git','-C',dest,'pull','--ff-only'], check=False)
    else:
        subprocess.run(['git','clone','--depth','1','--branch',branch,f'https://github.com/{repo}.git',dest], check=True)
    print('クローン成功:', dest)
except subprocess.CalledProcessError as e:
    print('git clone が失敗しました:', e)

# 編集可能インストール（失敗してもノートブックは継続）
try:
    subprocess.run([sys.executable, '-m', 'pip','install','-e', dest], check=True)
    print('pip install -e 完了')
except subprocess.CalledProcessError:
    print('pip install -e に失敗しました。setup.py/pyproject の有無を確認してください。')

# importability チェック
spec = importlib.util.find_spec('heretic')
print('heretic importable:', spec is not None)

# グローバル変数としてパスを保存して後続セルで再利用可能にする
HERETIC_PATH = dest
print('HERETIC_PATH set to', HERETIC_PATH)

# ガベージコレクション
gc.collect()


一部の依存をアップデートする必要があります。

In [ ]:
#@title Install dependencies & optimize for CUDA (自動実行)
# NOTE: このセルは上記の Clone & install セルの後に実行してください。
import subprocess, sys, os

# HERETIC_PATH が定義されていることを確認
try:
    heretic_path = HERETIC_PATH
except NameError:
    print('ERROR: HERETIC_PATH が定義されていません。先に Clone & install セルを実行してください。')
    heretic_path = None

if heretic_path and os.path.exists(heretic_path):
    print(f'Installing dependencies from {heretic_path}...')

    # 1. pyproject.toml からすべての依存をインストール
    try:
        print('\n[Step 1/4] Installing dependencies from pyproject.toml...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', heretic_path], check=True)
        print('✓ Dependencies installed')
    except subprocess.CalledProcessError as e:
        print(f'✗ Failed to install dependencies: {e}')

    # 2. PyTorch と torchvision を CUDA 121 向けにアップグレード
    try:
        print('\n[Step 2/4] Upgrading PyTorch & torchvision for CUDA 121...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'torch', 'torchvision',
                       '--index-url', 'https://download.pytorch.org/whl/cu121', '-q'], check=True)
        print('✓ PyTorch upgraded')
    except subprocess.CalledProcessError as e:
        print(f'✗ Failed to upgrade PyTorch: {e}')

    # 3. bitsandbytes をアップグレード
    try:
        print('\n[Step 3/4] Upgrading bitsandbytes...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'bitsandbytes', '-q'], check=True)
        print('✓ bitsandbytes upgraded')
    except subprocess.CalledProcessError as e:
        print(f'✗ Failed to upgrade bitsandbytes: {e}')

    # 4. transformers をアップグレード
    try:
        print('\n[Step 4/4] Upgrading transformers...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'transformers', '-q'], check=True)
        print('✓ transformers upgraded')
    except subprocess.CalledProcessError as e:
        print(f'✗ Failed to upgrade transformers: {e}')

    print('\n✓ All dependencies installed and optimized.')
else:
    print(f'ERROR: Path {heretic_path} does not exist.')

In [ ]:
# チェックポイント保存先の設定（Drive をマウントしている場合は Drive に、そうでなければ /content に保存）
import os, shutil
# MOUNT_DRIVE が定義されていなければ False を仮定
try:
    MOUNT_DRIVE
except NameError:
    MOUNT_DRIVE = False

if MOUNT_DRIVE:
    CHECKPOINT_DIR = '/content/drive/MyDrive/heretic/checkpoints'
else:
    CHECKPOINT_DIR = '/content/checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints will be saved to', CHECKPOINT_DIR)

# 簡単な同期例（必要に応じて手動で実行）
print('To sync checkpoints to Drive later (if MOUNT_DRIVE is True):')
print('  !rsync -av --progress /content/checkpoints/ /content/drive/MyDrive/heretic/checkpoints/')
print('Or use copy:')
print('  !cp -r /content/checkpoints/* /content/drive/MyDrive/heretic/checkpoints/')
